PIRVN — Evaluate Fine-Tuned Model

In [ ]:
# Test the fine-tuned price predictor against the test set.
import json
import re
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
test_path = Path("../phase2_curation/curated_data/test.json")

with open(test_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

texts = [item["text"] for item in test_data]
prices = [float(item["price"]) for item in test_data]

print(f"Loaded {len(texts)} test items")
print(f"Price range: {min(prices):,.0f} – {max(prices):,.0f} VND")
print(f"Mean price:  {np.mean(prices):,.0f} VND")

Option A: Test via Ollama (local GGUF)

In [ ]:
from litellm import completion
import os

MODEL = os.getenv("OLLAMA_MODEL", "pirvn-pricer")
BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

def predict_price_ollama(text):
    response = completion(
        model=f"ollama/{MODEL}",
        messages=[
            {"role": "system", "content": "You are a Vietnamese product price estimator. Respond with only the price number in VND."},
            {"role": "user", "content": f"San pham nay gia bao nhieu (VND)?\n\n{text}"},
        ],
        api_base=BASE_URL,
    )
    reply = response.choices[0].message.content.strip()
    match = re.search(r"\d+", reply.replace(",", "").replace(".", ""))
    return float(match.group()) if match else 0.0

Option B: Test via HuggingFace Spaces API

In [ ]:
import requests

HF_SPACE_URL = os.getenv("SPECIALIST_HF_SPACE", "https://your-username-pirvn-pricer.hf.space/api/predict")

def predict_price_hf(text):
    resp = requests.post(HF_SPACE_URL, json={"data": [text]}, timeout=60)
    return float(resp.json()["data"][0])

Run Evaluation

In [ ]:
# Choose which backend to use: predict_price_ollama or predict_price_hf
predict_fn = predict_price_ollama
N = 200

actuals = []
predictions = []
errors = []

for i, (text, price) in enumerate(tqdm(zip(texts[:N], prices[:N]), total=N, desc="Evaluating")):
    try:
        pred = predict_fn(text)
    except Exception as e:
        print(f"[{i}] Error: {e}")
        pred = 0.0
    actuals.append(price)
    predictions.append(pred)
    errors.append(abs(pred - price))

mae = np.mean(errors)
median_ae = np.median(errors)

print(f"\n{'='*40}")

In [ ]:
print(f"Evaluated {len(actuals)} items")
print(f"MAE:    {mae:>15,.0f} VND")
print(f"Median: {median_ae:>15,.0f} VND")
print(f"{'='*40}")
actuals_arr = np.array(actuals)
preds_arr = np.array(predictions)
rel_errors = np.abs(preds_arr - actuals_arr) / np.maximum(actuals_arr, 1)

# Color by relative error: green < 20%, orange < 40%, red >= 40%
colors = np.where(rel_errors < 0.20, "green", np.where(rel_errors < 0.40, "orange", "red"))

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(actuals_arr, preds_arr, c=colors, alpha=0.6, edgecolors="none", s=40)

mn = min(actuals_arr.min(), preds_arr.min())
mx = max(actuals_arr.max(), preds_arr.max())
ax.plot([mn, mx], [mn, mx], "k--", linewidth=1, label="y = x (perfect)")

def fmt_vnd(x, _):
    if x >= 1_000_000:
        return f"{x/1_000_000:.1f}M"
    elif x >= 1_000:
        return f"{x/1_000:.0f}K"
    return f"{x:.0f}"

import matplotlib.ticker as mticker
ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_vnd))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_vnd))

# Legend patches
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="green",  label="< 20% error"),
    Patch(facecolor="orange", label="20–40% error"),
    Patch(facecolor="red",    label="≥ 40% error"),
]
ax.legend(handles=legend_elements, loc="upper left")

ax.set_xlabel("Actual Price (VND)")
ax.set_ylabel("Predicted Price (VND)")
ax.set_title(f"Predicted vs Actual — MAE {mae:,.0f} VND")
plt.tight_layout()
plt.savefig("eval_scatter.png", dpi=120)
plt.show()
print("Saved eval_scatter.png")

In [ ]:
# Running average MAE as more items are evaluated
running_mae = np.cumsum(errors) / np.arange(1, len(errors) + 1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, len(running_mae) + 1), running_mae, color="steelblue", linewidth=1.5)
ax.axhline(mae, color="red", linestyle="--", linewidth=1, label=f"Final MAE: {mae:,.0f} VND")

ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_vnd))
ax.set_xlabel("Items Evaluated")
ax.set_ylabel("Running Average MAE (VND)")
ax.set_title("Error Convergence Over Test Set")
ax.legend()
plt.tight_layout()
plt.savefig("eval_running_mae.png", dpi=120)
plt.show()
print("Saved eval_running_mae.png")

Compare with Baselines

In [ ]:
baselines = [
    ("Constant (mean price)",         None),   # fill in after running constant baseline
    ("XGBoost",                        None),   # fill in after running XGBoost baseline
    ("GPT-4.1-nano (zero-shot)",       None),   # fill in after running GPT baseline
    ("Qwen3-4B base (zero-shot)",      None),   # fill in after running Qwen3 zero-shot
    ("PIRVN Fine-tuned (this model)",  mae),
]

print(f"{'Model':<35} {'MAE (VND)':>20}")
print("-" * 57)
for name, value in baselines:
    if value is None:
        display = "         — (not run)"
    else:
        display = f"{value:>20,.0f}"
    marker = " ◄" if name.startswith("PIRVN") else ""
    print(f"{name:<35} {display}{marker}")